# Feature Selection - Local Targets

Pearson r and AUC evaluated against **each stage's own yield target** (`stage['target']`).


## Section 1: Setup & Configuration

### 1.1 Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_colwidth', 60)

### 1.2 File Paths

In [ ]:
import os
from pathlib import Path

BASE_DIR   = Path(r'C:/Hackathon-GSK/Final_Submission/outputs/basetable')
CHARTS_DIR = Path(r'C:/Hackathon-GSK/Final_Submission/outputs/feature_selection/charts_v5')
OUTPUTS_DIR = Path(r'C:/Hackathon-GSK/Final_Submission/outputs/feature_selection')

CHARTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

### 1.3 SEROTYPE_CONFIG 


In [3]:
SEROTYPE_CONFIG = [
    ('IP1', 'ip1_basetable_v5.parquet', 'ST1'),
    ('IP2', 'ip2_basetable_v5.parquet', 'ST2'),
    ('IP3', 'ip3_basetable_v5.parquet', 'ST3'),
]

### 1.4 STAGE_CONFIG

In [4]:
STAGE_CONFIG = [
    {'name': 'clarif', 'feature_prefixes': ['clarif_'],
     'target': 'clarif_007 Clarif - Yield total [%]', 'target_type': 'local',
     'exclude_output_cols': True, 'output_exclude_patterns': [
         'Ag content', 'Protein by Lowry', 'Ag total',
         'Prot total', 'Yield', 'Total volume', '% elimination']},
    {'name': 'uf', 'feature_prefixes': ['UF_'],
     'target': 'UF_008 UF - Yield [%]', 'target_type': 'local',
     'exclude_output_cols': False, 'output_exclude_patterns': []},
    {'name': 'pg', 'feature_prefixes': ['PG_', 'pg_'],
     'target': 'PG_009 PG - Yield [%]', 'target_type': 'local',
     'exclude_output_cols': False, 'output_exclude_patterns': []},
    {'name': 'deae', 'feature_prefixes': ['DEAE_', 'deae_', 'ev_'],
     'target': 'GY_011 PSV - Global Yield total [%]', 'target_type': 'global',
     'exclude_output_cols': False, 'output_exclude_patterns': []},
    {'name': 'psv_global', 'feature_prefixes': ['PSV_', 'GY_'],
     'target': 'GY_011 PSV - Global Yield total [%]', 'target_type': 'global',
     'exclude_output_cols': True, 'output_exclude_patterns': []},
]

### 1.5 Drop Patterns & Audit Column Helper
Define column-name sub-strings that always indicate non-feature columns, plus a function to remove them.

In [5]:
DROP_PATTERNS = [
    'Corrected Parameter', 'Double check', '3rd check', 'Remark', 'NF_','prep_'
]
ID_COLS = ['Batch', 'Reference Date']

def drop_audit_cols(df):
    to_drop = [c for c in df.columns
               if any(pat in c for pat in DROP_PATTERNS)]
    df.drop(columns=to_drop, inplace=True, errors='ignore')
    return to_drop


### 1.6 ALL_YIELD_COLS

In [6]:
ALL_YIELD_COLS = [
    'clarif_Clarif - Yield Ag [%]',
    'clarif_007 Clarif - Yield total [%]',
    'UF_008 UF - Yield [%]',
    'PG_009 PG - Yield [%]',
    'PSV_010 PSV - Yield [%]',
    'GY_PSV - Global Yield [%]',
    'GY_011 PSV - Global Yield total [%]',
]

## Section 2:  Load & Validate All 3 Serotypes

### 2.1 Load CSVs

In [7]:
data = {}  # keyed by serotype label, e.g. 'IP1'

for label, fname, st_code in SEROTYPE_CONFIG:
    path = BASE_DIR / fname
    df = pd.read_parquet(path)
    data[label] = df

### 2.2 Drop Audit Columns & Preprocess Known Non-Numeric Fields
Remove audit-trail columns and transform column `DEAE - Stock duration pool PG` which is stored as `HH:MM:SS` into numeric.

In [8]:
def hms_to_min(s):
    try:
        parts = str(s).split(':')
        return int(parts[0]) * 60 + int(parts[1]) + int(parts[2]) / 60
    except Exception:
        return np.nan

for label in data:
    df = data[label]
    dropped = drop_audit_cols(df)
    print(f'{label}: dropped {len(dropped)} audit columns → {df.shape[1]} remain')

    dur_col = 'DEAE_DEAE - Stock duration pool PG'
    if dur_col in df.columns:
        df[dur_col] = df[dur_col].apply(hms_to_min)
        print(f'  Converted {dur_col} to minutes')

IP1: dropped 117 audit columns → 747 remain
  Converted DEAE_DEAE - Stock duration pool PG to minutes
IP2: dropped 104 audit columns → 741 remain
  Converted DEAE_DEAE - Stock duration pool PG to minutes
IP3: dropped 115 audit columns → 746 remain
  Converted DEAE_DEAE - Stock duration pool PG to minutes


### 2.3 Comparison Table

In [9]:
target_col = 'GY_011 PSV - Global Yield total [%]'
rows = []
for label, fname, st_code in SEROTYPE_CONFIG:
    df = data[label]
    rows.append({
        'Serotype': label,
        'Batches': len(df),
        'Cols after drop': df.shape[1],
    })

summary = pd.DataFrame(rows).set_index('Serotype')
print(summary.to_string())

          Batches  Cols after drop
Serotype                          
IP1            88              747
IP2            30              741
IP3            87              746


#### Functions - `get_candidate_features()`

Returns the list of valid, non-leaking numeric candidate features for a given stage.

Filtering pipeline applied in order:

- Step 1: Keep only columns whose prefix matches `stage['feature_prefixes']`

- Step 2: Remove any column in `ALL_YIELD_COLS`

- Step 3: Remove any column matching `DROP_PATTERNS`

- Step 4: For **clarif** stage: remove output-measurement columns

- Step 5: For **psv_global** stage: remove GY_ yield columns

- Step 6: Keep only columns with ≥ 5 non-NaN numeric values

The result is a feature-safe list that can be passed directly to any statistical test.


In [10]:
def get_candidate_features(df, stage):
    prefixes = stage['feature_prefixes']
    cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes)]
    cols = [c for c in cols if c not in ALL_YIELD_COLS]
    cols = [c for c in cols if not any(pat in c for pat in DROP_PATTERNS)]
    if stage.get('exclude_output_cols') and stage['name'] == 'clarif':
        out_pats = stage.get('output_exclude_patterns', [])
        cols = [c for c in cols if not any(p in c for p in out_pats)]
    if stage['name'] == 'psv_global':
        cols = [c for c in cols if not (c.startswith('GY_') and 'Yield' in c)]
    num_df = df[cols].apply(pd.to_numeric, errors='coerce')
    cols = [c for c in cols if num_df[c].notna().sum() >= 5]
    return cols

## Section 3: Feature Selection - Local Stage Targets

### 3.1 Initialise Result Lists
Create Three flat Python lists to accumulate every result row across all serotypes and stages.

In [11]:
pearson_rows = []   # collects: Serotype, Stage, Feature, Pearson_r, Abs_r, P_value

### 3.2 Pearson r - Chart Function

In [12]:
def plot_pearson_bar(df_slice, stage, serotype_name, mode_tag='local'):
    # Take the top-20 by absolute Pearson r from the provided DataFrame slice.
    top = df_slice.nlargest(20, 'Abs_r')
    if top.empty:
        return
    # Blue bar = positive correlation, tomato bar = negative correlation.
    colors = ['steelblue' if r >= 0 else 'tomato' for r in top['Pearson_r']]
    fig, ax = plt.subplots(figsize=(10, max(4, len(top) * 0.38)))
    ax.barh(range(len(top)), top['Pearson_r'], color=colors)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([f.split('_', 1)[-1][:52] for f in top['Feature']], fontsize=7)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Pearson r')
    ax.set_title(serotype_name + ' — ' + stage['name']
                 + ' Pearson r  (' + mode_tag + ')', fontsize=9)
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / (serotype_name + '_' + stage['name']
                + '_' + mode_tag + '_pearson.png'), bbox_inches='tight', dpi=100)
    plt.close(fig)

### 3.2 Pearson r - Collection Loop (Local Targets)

Iterates over every **serotype × stage** pair (3 × 5 = 15 combinations) and
computes Pearson r between each continuous feature and **`stage['target']`**.

Each valid feature-score pair is appended as a **flat dictionary** to `pearson_rows`.
After all 15 combinations are processed, the list is converted to `all_pearson_df`
in one call: `pd.DataFrame(pearson_rows)`.

In [13]:
# Outer loop: one pass per serotype.
for label, fname, st_code in SEROTYPE_CONFIG:
    df = data[label]

    # Inner loop: one pass per purification stage.
    for stage in STAGE_CONFIG:

        target = stage['target']

        # Retrieve valid, non-leaking numeric candidate features for this stage.
        cands      = get_candidate_features(df, stage)
        num_df     = df[cands].apply(pd.to_numeric, errors='coerce')
        cont_feats = [c for c in cands if num_df[c].nunique() > 2]  # continuous only

        # Feature-level loop: compute Pearson r for each continuous candidate.
        for feat in cont_feats:
            # Paired drop: use only rows where BOTH feature and target are non-NaN.
            valid = df[[feat, target]].apply(pd.to_numeric, errors='coerce').dropna()
            if len(valid) < 5:    # need at least 5 observations for a valid correlation
                continue
            try:
                res = scipy.stats.pearsonr(valid[feat], valid[target])
                r, p = float(res.statistic), float(res.pvalue)
            except Exception:
                continue
            # Append one flat dictionary row — no nesting.
            pearson_rows.append({
                'Serotype': label,  'Stage': stage['name'],  'Feature': feat,
                'Pearson_r': round(r, 4),  'Abs_r': round(abs(r), 4),  'P_value': round(p, 4),
            })

# Convert the flat list to DataFrame
all_pearson_df = pd.DataFrame(pearson_rows)
print('\nall_pearson_df: ' + str(all_pearson_df.shape[0]) + ' rows × '
      + str(all_pearson_df.shape[1]) + ' cols')
print(all_pearson_df.head(200).to_string())


all_pearson_df: 1798 rows × 6 cols
    Serotype   Stage                                          Feature  Pearson_r  Abs_r  P_value
0        IP1  clarif                          clarif_P142-Cells count     -0.016  0.016    0.880
1        IP1  clarif                              clarif_Nr fermentor      0.080  0.080    0.461
2        IP1  clarif     clarif_Duration cells pool transfer [Minute]     -0.063  0.063    0.558
3        IP1  clarif                          clarif_D0-Viability [%]      0.081  0.081    0.453
4        IP1  clarif                            clarif_D0-Cells count      0.045  0.045    0.676
5        IP1  clarif                         clarif_D1 8h-Cells count      0.045  0.045    0.678
6        IP1  clarif                    clarif_D2 8h-Cells count Cell     -0.106  0.106    0.327
7        IP1  clarif           clarif_Vol medium transf day 3 [Liter]      0.038  0.038    0.725
8        IP1  clarif                     clarif_D3 AM-cell count Cell      0.047  0.047    

In [14]:
# Save Pearson bar chart for each serotype × stage combination.
for (sero, sn), grp in all_pearson_df.groupby(['Serotype', 'Stage'], sort=False):
    stage_obj = next(s for s in STAGE_CONFIG if s['name'] == sn)
    plot_pearson_bar(grp, stage_obj, sero, mode_tag='local')
print('Pearson bar charts saved.')

Pearson bar charts saved.


### 3.3 Select Top Regression Features

Applies three sequential Pandas operations to produce `top_reg_df`:

1. Filter: Keep only statistically suggestive correlations (p < 0.10)

2. Sort: Descending by |r| within each Serotype × Stage group

3. Top-10 per group: Keep the ten strongest features per group using groupby + head.

A `Rank` column is added (1 = strongest). No Python loops or `.to_dict('records')` involved.

In [15]:
# Step 1 - Filter
sig_pearson_df = all_pearson_df[all_pearson_df['P_value'] < 0.10].copy()
print('Features passing p < 0.10: ' + str(len(sig_pearson_df))
      + '  (from ' + str(len(all_pearson_df)) + ' total)')

# Step 2 - Sort
sig_pearson_df = sig_pearson_df.sort_values(
    ['Serotype', 'Stage', 'Abs_r'], ascending=[True, True, False]
)

# Step 3 - Top-10
top_reg_df = (sig_pearson_df
              .groupby(['Serotype', 'Stage'], group_keys=False)
              .head(10)
              .reset_index(drop=True))

# Step 4 - Rank
top_reg_df['Rank'] = (top_reg_df
                      .groupby(['Serotype', 'Stage'])
                      .cumcount() + 1)

print('top_reg_df shape: ' + str(top_reg_df.shape) + '  (max 10 rows per group)')
print(top_reg_df[['Serotype', 'Stage', 'Feature', 'Pearson_r', 'Rank']].head(5).to_string())

# Full list — all features sorted by |r| per group, no p-value filter applied
all_pearson_sorted = all_pearson_df.sort_values(
    ['Serotype', 'Stage', 'Abs_r'], ascending=[True, True, False]
).reset_index(drop=True)

print('\nall_pearson_sorted shape: ' + str(all_pearson_sorted.shape) + '  (all features, all stages)')
print(all_pearson_sorted[['Serotype', 'Stage', 'Feature', 'Pearson_r', 'Abs_r', 'P_value']].head(10).to_string())

Features passing p < 0.10: 351  (from 1798 total)
top_reg_df shape: (99, 7)  (max 10 rows per group)
  Serotype   Stage                                     Feature  Pearson_r  Rank
0      IP1  clarif         clarif_Clarif 1 - Flow 340L (L/min)      0.199     1
1      IP1    deae  DEAE_DEAE - equilibration conducti [mS/cm]     -0.265     1
2      IP1    deae                   deae_sensor_MT_I2_OUT_max      0.220     2
3      IP1    deae                deae_child2_xto_prompt__17_1      0.216     3
4      IP1    deae                 deae_child2_xto_prompt__9_1     -0.196     4

all_pearson_sorted shape: (1798, 6)  (all features, all stages)
  Serotype   Stage                                          Feature  Pearson_r  Abs_r  P_value
0      IP1  clarif              clarif_Clarif 1 - Flow 340L (L/min)      0.199  0.199    0.063
1      IP1  clarif      clarif_001 Ag cont Elisa virus cult [DU/ml]     -0.176  0.176    0.100
2      IP1  clarif     clarif_D3 8h-clarif. 1 Volume before [Liter]  

## Section 4: Cross-Serotype Comparison & Save

Aggregates Pearson r results from Section 3 across all three serotypes to identify
features consistently selected across IP1, IP2, and IP3 — the most biologically robust signals.

### 4.1 Cross-Serotype Regression Comparison

For each stage, pivots top regression selected features df `top_reg_df` to a wide-format table

Rows = features, columns = IP1 / IP2 / IP3. A `✓` in "In all 3?" marks features
that appear in every serotype.

In [16]:
# For each stage, pivot top_reg_df to a wide table: rows=features, cols=serotypes.
for stage in STAGE_CONFIG:
    sn = stage['name']

    # Slice: only this stage's selected features.
    stage_df = top_reg_df[top_reg_df['Stage'] == sn].copy()
    if stage_df.empty:
        print(sn + ': no features selected')
        continue

    # Pivot: Feature as index, Serotype as columns, Pearson_r as values.
    pivot = stage_df.pivot_table(
        index='Feature', columns='Serotype', values='Pearson_r', aggfunc='first'
    )

    # Add missing serotype columns (NaN = feature not selected for that serotype).
    for col in ['IP1', 'IP2', 'IP3']:
        if col not in pivot.columns:
            pivot[col] = float('nan')
    pivot = pivot[['IP1', 'IP2', 'IP3']]

    # Mark features present in all three serotypes.
    pivot['In all 3?'] = pivot.notna().all(axis=1).map({True: '✓', False: '—'})

    # Shorten feature names: strip prefix, truncate to 55 chars.
    pivot.index = [f.split('_', 1)[-1][:55] for f in pivot.index]
    pivot.index.name = 'Feature'
    pivot.columns.name = None

    print('--- ' + sn + ' ---')
    print(pivot.to_string())
    print()

--- clarif ---
                                        IP1    IP2    IP3 In all 3?
Feature                                                            
001 Ag cont Elisa virus cult [DU/ml]    NaN    NaN -0.413         —
014 Protein Lowry virus cult [µg/ml]    NaN    NaN -0.260         —
031 Clarif 1-Flow 300L (L/min)          NaN    NaN  0.193         —
Clarif 1 - Flow 340L (L/min)          0.199    NaN  0.182         —
Clarif 2 - Duration min [Minute]        NaN    NaN -0.193         —
D0-Cells count                          NaN    NaN  0.284         —
D3 AM-cell count Cell                   NaN  0.332    NaN         —
D4 AM-cell count Cell                   NaN  0.361    NaN         —
D5 8h-Cells count                       NaN    NaN  0.253         —
Duration cells pool transfer [Minute]   NaN    NaN  0.222         —
Nr fermentor                            NaN  0.347    NaN         —
Perf 1vol1/2-Vol medium tr. [Liter]     NaN -0.355    NaN         —
Vol medium transf day 3 [Liter]  

### 4.2 Save Results

In [17]:
top_reg_df.to_csv(OUTPUTS_DIR / 'regression_local_selected_features_v5.csv', index=False)
all_pearson_sorted.to_csv(OUTPUTS_DIR / 'all_pearson_scores_local_v5.csv', index=False)
print(f'Saved: regression_local_selected_features_v5.csv  ({len(top_reg_df)} rows)')
print(f'Saved: all_pearson_scores_local_v5.csv            ({len(all_pearson_sorted)} rows)')

Saved: regression_local_selected_features_v5.csv  (99 rows)
Saved: all_pearson_scores_local_v5.csv            (1798 rows)
